In [2]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics.pairwise import euclidean_distances

In [3]:
#feature data
df = pd.read_csv("laptops_Featured.csv")

# Load model 
artifacts = joblib.load("model_artifacts.joblib")

# Extract objects
model = artifacts["model"]
label_encoders = artifacts["label_encoders"]
feature_columns = artifacts["feature_columns"]

print("Dataset Shape :", df.shape)
print("Features Used :", feature_columns)

Dataset Shape : (1014, 16)
Features Used : ['Brand', 'CPU_Score', 'GPU_Score', 'RAM_GB', 'Storage_GB', 'Refresh_Hz', 'Resolution_Score']


In [4]:
def recommend_laptops(
    brand,
    cpu_score,
    gpu_score,
    ram_gb,
    storage_gb,
    refresh_hz,
    resolution_score,
    top_n=3
):

    # Encode brand
    encoded_brand = label_encoders["Brand"].transform([brand])[0]

    # User input
    user_input = pd.DataFrame([{
        "Brand": encoded_brand,
        "CPU_Score": cpu_score,
        "GPU_Score": gpu_score,
        "RAM_GB": ram_gb,
        "Storage_GB": storage_gb,
        "Refresh_Hz": refresh_hz,
        "Resolution_Score": resolution_score
    }])

    # Predict performance score
    predicted_score = model.predict(user_input)[0]

    # Dataset for recommendation
    recommendation_data = df[feature_columns].copy()

    recommendation_data["Brand"] = label_encoders["Brand"].transform(
        recommendation_data["Brand"]
    )

    # Calculate Euclidean distance
    distances = euclidean_distances(
        user_input,
        recommendation_data
    )

    # Get nearest laptops
    nearest = distances.argsort()[0][:top_n]

    recommendations = df.iloc[nearest].copy()

    return recommendations, round(predicted_score, 2)

In [5]:
result, predicted_score = recommend_laptops(
    brand="ASUS",
    cpu_score=8,
    gpu_score=7,
    ram_gb=16,
    storage_gb=1024,
    refresh_hz=165,
    resolution_score=3
)
print(f"Predicted Performance Score : {predicted_score:.2f}/100")

result[[
    "Brand",
    "Product_Name",
    "Processor",
    "GPU",
    "RAM",
    "Price",
    "Performance_Score"
]]

Predicted Performance Score : 66.38/100


,Brand,Product_Name,Processor,GPU,RAM,Price,Performance_Score
343,ASUS,ROG Zephyrus G15 GA503RM-HQ007W,AMD Ryzen 7-6800HS,"NVIDIA GeForce RTX™ 3060, 6GB",16GB Ram,44999,64.69
425,ASUS,TUF Gaming F15 FX507ZM-HQ013W,Intel Core I7-12700H,"NVIDIA® GeForce RTX™ 3060 Laptop GPU, 1752MHz ...",16GB,44449,64.69
230,ASUS,ROG Strix G16 G614JV-N3487W,Intel Core i7-13650HX,Nvidia GeForce RTX 4060 8GB,16GB DDR5-4800 SO-DIMM,62999,69.94


In [6]:
print(result.columns.tolist())

['Brand', 'Product_Name', 'Processor', 'GPU', 'RAM', 'Storage', 'Display_Resolution', 'Refresh_Rate', 'Price', 'RAM_GB', 'Storage_GB', 'Refresh_Hz', 'CPU_Score', 'GPU_Score', 'Resolution_Score', 'Performance_Score']


In [7]:
result, predicted_score = recommend_laptops(
    brand="MSI",
    cpu_score=10,
    gpu_score=10,
    ram_gb=32,
    storage_gb=2048,
    refresh_hz=240,
    resolution_score=5
)

print(f"Predicted Performance Score : {predicted_score:.2f}/100")

result[[
    "Brand",
    "Product_Name",
    "Price",
    "Performance_Score"
]]

Predicted Performance Score : 93.25/100


,Brand,Product_Name,Price,Performance_Score
273,MSI,Raider GE78 HX 14VIG,180449,92.25
198,MSI,Raider 18 HX A14VIG,176499,92.25
466,NaN,ROG Strix Scar 17 SE. G733C-XS97,114999,82.50


In [8]:
print("=" * 60)
print("Laptop Recommendation Engine Completed Successfully!")
print("=" * 60)

Laptop Recommendation Engine Completed Successfully!
